# BOT de Portafolios v2. Inicio en Python y pandas
Primero explicar el mandato y la diferencia entre logs y retorno de cartera. El cuaderno valida la entrada sintética y permite probar cotizaciones; no entrega resueltos optimización, BL ni rebalanceo. La consulta Yahoo se activa aparte y sus datos no se versionan ni distribuyen sin derechos.

In [ ]:
from pathlib import Path
import json, hashlib, platform
import numpy as np
import pandas as pd
ROOT=Path.cwd()
if not (ROOT/'data').is_dir(): ROOT=ROOT.parent
file=ROOT/'data'/'precios_sinteticos_20.csv'
if not file.is_file(): raise FileNotFoundError('Abra el cuaderno desde la carpeta del paquete o cuadernos/.')
metadata=json.loads((ROOT/'data'/'precios_sinteticos_metadata.json').read_text())
assert hashlib.sha256(file.read_bytes()).hexdigest()==metadata['sha256']
prices=pd.read_csv(file,parse_dates=['fecha']).set_index('fecha').sort_index()
assert prices.index.is_unique and prices.shape==(600,20)
assert np.isfinite(prices.to_numpy()).all() and (prices>0).all().all()
print({'source':metadata['tipo'],'calendar':metadata['calendario'],'python':platform.python_version(),'pandas':pd.__version__,'numpy':np.__version__})

## Rendimientos y matriz
Reconstruir una entrada de covarianza a mano antes de interpretar la matriz. La máscara se aplica después de verificar el calendario y calcular intervalos válidos; no utilizar esta receta para atravesar huecos de una serie real.

In [ ]:
simple=prices.pct_change(fill_method=None)
logs=np.log(prices/prices.shift(1))
mask=simple.notna().all(axis=1)
R=simple.loc[mask];g=logs.loc[mask]
assert len(R)==599
np.testing.assert_allclose(np.expm1(g),R,atol=1e-12)
Sigma=R.cov(ddof=1);correlation=R.corr();mu=R.mean()
assert Sigma.index.equals(Sigma.columns) and Sigma.index.equals(mu.index)
np.testing.assert_allclose(Sigma,Sigma.T,atol=1e-12)
print({'n':len(R),'assets':len(mu),'min_eigenvalue':float(np.linalg.eigvalsh(Sigma).min())})

## Consulta Yahoo optativa
Revisar documentación actual de yfinance y condiciones de Yahoo; instalar las dependencias faltantes en un entorno aislado y registrar sus versiones. La celda hace dos consultas de barras de un minuto: no es streaming tick a tick, no garantiza ausencia de retraso y la última barra puede estar incompleta. No se guardan cotizaciones en archivos ni se usan esas barras aquí para optimizar.

In [ ]:
CONSULTAR_YAHOO=False
TICKER='AAPL'  # Solo prueba de conexión, no selección de inversión.
SEGUNDOS_ENTRE_CAPTURAS=30
if CONSULTAR_YAHOO:
    import time
    from datetime import datetime, timezone
    import yfinance as yf
    import matplotlib.pyplot as plt
    from IPython.display import display, clear_output
    ticker=yf.Ticker(TICKER)
    for attempt in range(2):
        if attempt: time.sleep(SEGUNDOS_ENTRE_CAPTURAS)
        try:
            bars=ticker.history(period='1d',interval='1m',auto_adjust=False,actions=True,repair=False,prepost=False,timeout=10,raise_errors=True)
            if bars.empty or 'Close' not in bars: raise ValueError('Sin barras utilizables.')
            close=bars['Close'].dropna()
            if close.empty or not np.isfinite(close).all() or (close<=0).any(): raise ValueError('Precios inválidos.')
            info=ticker.get_history_metadata()
            currency=info.get('currency') or 'no informada'
            clear_output(wait=True)
            print({'source':'Yahoo vía yfinance','ticker':TICKER,'currency':currency,'received_at':datetime.now(timezone.utc).isoformat(),'latest_bar_start':str(close.index[-1]),'delay':'desconocido','last_bar_may_be_incomplete':True})
            fig,ax=plt.subplots(figsize=(9,4))
            ax.plot(close.index,close.to_numpy())
            ax.set_title(f'{TICKER}, cierre de barras de 1 minuto, captura {attempt+1}')
            ax.set_xlabel('Hora del índice del proveedor');ax.set_ylabel(f'Precio ({currency})')
            fig.autofmt_xdate();display(fig);plt.close(fig)
        except Exception as exc:
            raise RuntimeError('Consulta fallida; no se reemplaza por un fixture. Revise red, permisos y versión.') from exc
else:
    print('Yahoo no consultado. Laboratorio sintético, no prueba de mercado.')

## Próxima decisión del estudiante
Definir objetivo, rf, restricciones, ventana y resultado esperado del caso de dos activos. Pedir al tutor una prueba y el cambio mínimo, después explicar por qué cotización, estimación y rebalanceo tienen relojes diferentes.